In [1]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import torch

model_id = "Qwen/Qwen2.5-VL-7B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

min_pixels = 256 * 28 * 28
max_pixels = 512 * 28 * 28 # Обмеження, щоб модель не "зависала"
processor = AutoProcessor.from_pretrained(model_id, min_pixels=min_pixels, max_pixels=max_pixels)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

c:\UnProg\DIPLOMA\venv_diploma\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 729/729 [00:06<00:00, 104.25it/s]


In [2]:
import json

def load_onomatopoeia_dictionary(file_path='manhwa_onomatopoeia.json'):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def get_rag_context(dictionary, category_filter=None):
    """
    Формує RAG-контекст зі словника.
    category_filter — список категорій для фільтрації (або None = всі).
    """
    context_parts = ["Base of approved translations:"]
    for category in dictionary['categories']:
        if category_filter and category['label'] not in category_filter:
            continue
        context_parts.append(f"\nCategory: {category['label']} ({category['description']})")
        for entry in category['entries']:
            ukrainian = ', '.join(entry['ukrainian'])
            context_parts.append(
                f"  - {entry['korean']} ({entry['transliteration']}): {ukrainian} — {entry['meaning']}"
            )
    return "\n".join(context_parts)

In [3]:
import chromadb
from sentence_transformers import SentenceTransformer

# 1. Ініціалізація бази (локально)
dictionary = load_onomatopoeia_dictionary()
# Тепер база буде зберігатися в папці 'chroma_db' у твоєму проекті
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="onomatopoeia")

# ВИПРАВЛЕННЯ 2: багатомовна модель — розуміє корейську, українську, англійську
embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# ВИПРАВЛЕННЯ 3: upsert замість add — безпечно при повторному запуску
for cat in dictionary['categories']:
    for entry in cat['entries']:
        # ВИПРАВЛЕННЯ 4: індексуємо англійський опис + значення (те, що шукатиме VLM)
        text_to_index = f"{entry['meaning']} {cat['label']} {entry['transliteration']}"
        embedding = embed_model.encode(text_to_index).tolist()
        
        collection.upsert(                          # ← upsert, не add
            ids=[entry['korean']],
            embeddings=[embedding],
            metadatas=[{
                "ukrainian": ", ".join(entry['ukrainian']),
                "meaning": entry['meaning'],
                "category": cat['label'],
                "visual_hint": cat.get('visual_hint', '')
            }],
            documents=[text_to_index]
        )

print(f"✅ Проіндексовано записів: {collection.count()}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10968.89it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Проіндексовано записів: 27


In [4]:
# ====== КЛІТИНКА 5б: Функція пошуку ======

def get_precise_context(vlm_description: str, n_results: int = 3) -> str:
    """
    Пошук у ChromaDB по англійському опису від VLM.
    Повертає рядок з кандидатами для фінального промпту.
    """
    query_vector = embed_model.encode(vlm_description).tolist()
    results = collection.query(query_embeddings=[query_vector], n_results=n_results)
    
    context = "Reference candidates (Korean → Ukrainian):\n"
    for i in range(len(results['ids'][0])):
        meta = results['metadatas'][0][i]
        context += (
            f"- {results['ids'][0][i]} "
            f"({meta['meaning']}): {meta['ukrainian']}\n"
        )
    return context

In [5]:
from qwen_vl_utils import process_vision_info
import torch

def ask_model_simple(image_path, prompt, model, processor):
    """
    Універсальна функція для отримання відповіді від VLM.
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    # 1. Підготовка вхідних даних (текстовий шаблон та обробка зображення)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    # 2. Генерація відповіді
    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=128, # Достатньо для опису або короткої відповіді
            do_sample=False,    # Greedy decoding для стабільності результатів
            temperature=0       # Примусове дотримання логіки
        )

    # 3. Декодування результату (відрізаємо вхідний промпт)
    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    return output_text[0].strip()

In [6]:
# ====== КЛІТИНКА 7: Фінальний pipeline ======

def translate_with_vector_rag(image_path, model, processor):
    
    # КРОК 1: VLM описує зображення англійською
    # (англійська — бо embed_model краще працює з англійським запитом)
    description_prompt = (
        "Describe ONLY the visual action and sound style in this manga panel. "
        "Focus on: type of motion (sharp/flowing), intensity (loud/quiet), "
        "and what physical action is happening. Answer in English, 1-2 sentences."
    )
    visual_description = ask_model_simple(image_path, description_prompt, model, processor)
    print(f"[Step 1] VLM опис: {visual_description}")
    
    # КРОК 2: Векторний пошук кандидатів у ChromaDB
    precise_context = get_precise_context(visual_description)
    print(f"[Step 2] Кандидати:\n{precise_context}")
    
    # КРОК 3: Фінальний вибір
    # передаємо і опис, і кандидатів, просимо ОБРАТИ, а не перекладати
    final_prompt = (
        f"The action in this image was described as: '{visual_description}'\n\n"
        f"{precise_context}\n"
        "Choose the single most appropriate Ukrainian sound effect from the candidates above. "
        "Output ONLY the Ukrainian word, nothing else.\n",
    )
    
    result = ask_model_simple(image_path, final_prompt, model, processor)
    print(f"[Step 3] Результат: {result}")
    return result

In [7]:
translate_with_vector_rag("coco_converted/val/images/252.jpg", model, processor)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[Step 1] VLM опис: The character's back is shown in a flowing motion as they stand at the doorway, gazing out into the quiet night. The sound is quiet, almost a whisper, emphasizing the stillness and contemplation of the moment.
[Step 2] Кандидати:
Reference candidates (Korean → Ukrainian):
- 똑똑 (Стукіт у двері): ТУК-ТУК
- 끼익 (Скрип дверей або гальм): РИП, СКРИП
- 확 (Різкий рух, ривок, спалах): ВЖУХ, ПУРХ

[Step 3] Результат: РИП


'РИП'